In [17]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

In [18]:
# Helpers
def theta_nominal():
    T_REF = 328.15
    k_1_ref = 8.84e-3 * 60 * 0.5 # 1/min
    k_2_ref = k_1_ref / 8
    theta_1 = k_1_ref * np.exp(E_1 / (R_GAS * T_REF))
    theta_2 = k_2_ref * np.exp(E_2 / (R_GAS * T_REF))

    return theta_1, theta_2

In [19]:
# Global Constants
E_1 = 48_000 # J/mol (activation energy first reaction)
E_2 = 55_000 # J/mol (activation energy second reaction)
R_GAS = 8.314 # J/K/mol
th_1, th_2 = theta_nominal()

In [20]:
# Calculation of arrhenius prefactors (T-dependent)

def compute_k(T_1, T_2):
    k_1_R1 = th_1 * np.exp(- E_1 / (R_GAS * T_1))
    k_2_R1 = th_2 * np.exp(- E_2 / (R_GAS * T_1))
    k_1_R2 = th_1 * np.exp(- E_1 / (R_GAS * T_2))
    k_2_R2 = th_2 * np.exp(- E_2 / (R_GAS * T_2))

    return (k_1_R1, k_1_R2), (k_2_R1, k_2_R2)

In [21]:
# Mixer
def M1(rho, c_R, c_F):
    c_A_R, c_B_R, c_C_R = c_R[0], c_R[1], c_R[2]
    c_A_F, c_B_F, c_C_F = c_F[0], c_F[1], c_F[2]
    c_A_alpha = rho * c_A_R + (1 - rho) * c_A_F
    c_B_alpha = rho * c_B_R + (1 - rho) * c_B_F
    c_C_alpha = rho * c_C_R + (1 - rho) * c_C_F

    return c_A_alpha, c_B_alpha, c_C_alpha

In [22]:
# CSTR1
def R1(c_alpha, k_R1, tau_1):
    k_1_R1, k_2_R1 = k_R1[0], k_R1[1]
    c_A_alpha, c_B_alpha, c_C_alpha = c_alpha[0], c_alpha[1], c_alpha[2]

    c_A_R1 = c_A_alpha / (1 + k_1_R1 * tau_1)
    c_B_R1 = (c_B_alpha + k_1_R1 * c_A_R1 * tau_1) / (1 + k_2_R1 * tau_1)
    c_C_R1 = c_C_alpha + k_2_R1 * c_B_R1 * tau_1
    
    return  c_A_R1, c_B_R1, c_C_R1

In [23]:
# CSTR2
def R2(c_R1, k_R2, tau_2):
    c_A_R1, c_B_R1, c_C_R1 = c_R1[0], c_R1[1], c_R1[2]
    k_1_R2, k_2_R2 = k_R2[0], k_R2[1]

    c_A_R2 = c_A_R1 / (1 + k_1_R2 * tau_2)
    c_B_R2 = (c_B_R1 + k_1_R2 * c_A_R2 * tau_2) / (1 + k_2_R2 * tau_2)
    c_C_R2 = c_C_R1 + k_2_R2 * c_B_R2 * tau_2

    return c_A_R2, c_B_R2, c_C_R2

In [24]:
# Splitter
def S1(c_R2):
    # NOTE: rho: recycle ratio, dens: density
    c_A_R2, c_B_R2, c_C_R2 = c_R2[0], c_R2[1], c_R2[2]
    M_A, M_B, M_C = 78.11, 112.56, 147.00 # g/mol
    dens_A, dens_B, dens_C = 0.88e6, 1.11e6, 1.30e6 # g/m3
    s_A, s_B, s_C = 0.96, 0.15, 0.01 # these values are for illustration
    summands = [M_A / dens_A  * s_A * c_A_R2, M_B / dens_B  * s_B * c_B_R2, M_C / dens_C  * s_C * c_C_R2]
    rho = sum(summands)
    assert rho >= 0 and rho < 1
    c_A_R = s_A * c_A_R2 / rho
    c_B_R = s_B * c_B_R2 / rho
    c_C_R = s_C * c_C_R2 / rho

    return (c_A_R, c_B_R, c_C_R), rho

In [36]:
def get_informed_guess():
    """Make informed initial guess by finding nearest feasible configuration
    First value: is informed guess available (NONE if no feasible points found yet),
    Second value: rho_ig,
    Third value: c_R_ig"""
    return None, None, None

In [37]:
def is_converged(c_R_real_guess, rho_real_guess):
    """Takes in tuples of real and guessed values for c_R 
    (recycle stream concentration), and rho (recycle ratio)
    and compares them."""
    (c_A_R_real, c_B_R_real, c_C_R_real), (c_A_R_guess, c_B_R_guess, c_C_R_guess) = c_R_real_guess
    rho_real, rho_guess = c_R_real_guess
    real = np.array([c_A_R_real, c_B_R_real, c_C_R_real, rho_real])
    guess = np.array([c_A_R_guess, c_B_R_guess, c_C_R_guess])
    diff = np.abs(real - guess) # abs is safeguarding only
    diff_norm = diff / max(abs(real), 1e-8)
    max_vio = diff_norm.max()

    return max_vio

In [29]:
# Simulate forward (to be called by Nested Sampler)
def sim_forward(T_1, tau_1, T_2, tau_2):
    c_F = 11.28, 0, 0 # mol/L (pure benzene feed)
    # initial guesses (ig)
    inf_guess = get_informed_guess()
    rho_ig = 0.3 if not inf_guess[0] else inf_guess[1]
    c_R_ig = (5, 2, 1) if not inf_guess[0] else inf_guess[2]

    # k need fresh computatation at each configuration (T-dependent)
    k_R1, k_R2 = compute_k(T_1, T_2)

    c_alpha = M1(rho=rho_ig, c_R=c_R_ig, c_F=c_F)
    c_R1 = R1(c_alpha=c_alpha, k_R1=k_R1, tau_1=tau_1)
    c_R2 = R2(c_R1=c_R1, k_R2=k_R2, tau_2=tau_2)
    c_R, rho = S1(c_R2=c_R2)
    is_converged((c_R, c_R_ig), (rho, rho_ig))
